<div style="background: linear-gradient(120deg, #102a43 0%, #176b87 52%, #2a9d8f 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 5px 20px rgba(16,42,67,0.28);">
    <img src='Figures/iteso.png' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.22);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.45); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3; letter-spacing: 0.01em;">Examen Segundo Parcial Versión C</h1>
        <h3 style="margin: 0 0 8px 0; color: rgba(255,255,255,0.86); font-weight: normal; font-size: 1.05em;">Laboratorio de Procesamiento de Datos</h3>
        <div style="color: rgba(255,255,255,0.78); font-size: 0.95em;">Fecha: 05/11/2026</div>
    </div>
</div>


<div style="border-left: 4px solid #176b87; padding: 14px 18px; margin: 8px 0 18px 0;">
    <strong style="color: #102a43;">Nombre del alumno:</strong>
    <span> </span>
</div>



### Instrucciones
1. Resuelve individualmente los **tres ejercicios (100 puntos)** en este notebook. Agrega celdas de código y Markdown para tus procedimientos, resultados y justificaciones.
2. La entrada es `datos/student_examen.xlsx`, hoja `estudiantes`. En el inciso 1.1 debes leerla, construir `df_original` y exportarla a `salidas/student_extraido.csv` antes de procesarla. Usa `DATOS` y `SALIDA` de la configuración. No modifiques la entrada ni uses el objetivo para imputar o generar predictores.
3. Conserva `df_original` intacto y trabaja sobre copias. Todas las operaciones deben ser reproducibles. 
4. Las gráficas deben tener título y nombres de ejes. Acompaña las tablas y figuras con interpretaciones breves, una gráfica sin explicación no responde al inciso.
5. Entrega el notebook resuelto, la carpeta `Figures\`, `datos\`, `salidas\` en un archivo comprimido como se muestra en las instrucciones de CANVAS.

**Nota:** El notebook debe ejecutarse completo con **Restart & Run All** sin errores.  

## Conjunto de de Datos para el Examen: 
## Student Performance

El conjunto describe estudiantes de dos escuelas portuguesas. Este conjunto de datos contiene **649 filas del curso de lengua portuguesa**. Se tienen ocho variables predictoras y una  variable objetivo. El conjunto de datos se encuentra en el libro Excel `student_examen.xlsx`, hoja **`estudiantes`**, con encabezados en la primera fila y **nueve columnas**.

| Columna | Significado |
|---|---|
| `age` | Edad en años cumplidos |
| `absences` | Número de ausencias escolares, 0 es un valor válido |
| `studytime` | Nivel de tiempo semanal de estudio: 1 = menos de 2 horas; 2 = de 2 a 5; 3 = de 5 a 10; 4 = más de 10 |
| `Mjob` | Ocupación de la madre: `teacher`, `health`, `services`, `at_home`, `other` |
| `reason` | Motivo de elección de escuela: `home`, `reputation`, `course`, `other` |
| `higher` | Desea cursar educación superior (`yes`, `no`) |
| `internet` | Tiene acceso a internet en casa (`yes`, `no`) |
| `sex` | Categoría registrada de sexo (`F`, `M`) |
| `passed` | **Variable objetivo:** `yes` si la calificación final es al menos 10/20, `no` en caso contrario |


In [ ]:
# Configuración (no modificar)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

DATOS = "datos"
SALIDA = "salidas"
os.makedirs(SALIDA, exist_ok=True)
ENTRADA = os.path.join(DATOS, "student_examen.xlsx")

print("Archivo de entrada:", ENTRADA)
# La lectura y la creación de df_original y df corresponden al inciso 1.1.

---
## Ejercicio 1: Exploración, distribución y tratamiento de faltantes (30 pts)

**1.1 (5 pts).** Lee `ENTRADA` con `pd.read_excel`, seleccionando la hoja `estudiantes` y usando `pd.read_excel` o `ExcelFile` de pandas. Construye `df_original` con 649 filas y las nueve columnas descritas, considera las celdas vacías y `?` como datos faltantes, sin imputarlos. Exporta `salidas/student_extraido.csv` sin índice, con UTF-8 y celdas vacías para faltantes. Comprueba que una nueva lectura del CSV (`df`) preserve valores, faltantes y orden de filas (los tipos pueden variar). Trabaja después sobre `df = df_original.copy(deep=True)`. Construye una tabla con tipo de almacenamiento, tipo semántico (cuantitativo discreto/continuo, categórico nominal/ordinal o binario), número de valores distintos, faltantes y porcentaje de faltantes de cada columna. Identifica el objetivo y reporta duplicados completos sin eliminarlos automáticamente. Explica por qué `studytime` no equivale a un número de horas.

**1.2 (10 pts).** Para `age` y `absences`, reporta media, mediana, desviación estándar y cuartiles, genera histogramas y diagramas de caja con los datos observados. Cuenta posibles atípicos con límites `Q1 - 1.5*IQR` y `Q3 + 1.5*IQR`. Explica asimetría, concentración y por qué cero ausencias no representa un dato faltante. 

**1.3 (5 pts).** Crea `df_mediana`: imputa las dos columnas con faltantes numéricos mediante su mediana y `Mjob` y `reason` mediante la moda. Si hay empate en la moda, utiliza la primera categoría en orden alfabético. Reporta los valores utilizados y cuántas celdas imputaste. No alteres las celdas observadas.

**1.4 (10 pts).** En otra copia, compara esa imputación numérica con `KNNImputer(n_neighbors=5)`, usando únicamente `age` y `absences`. Antes de KNN, lleva ambas columnas a una escala comparable usando las medias y desviaciones de sus valores observados (estandarización), después recupera sus unidades originales y sustituye únicamente las celdas faltantes. Compara mediana y desviación de cada columna tras ambos métodos y explica una limitación de cada uno, incluyendo la posibilidad de obtener valores fraccionarios en variables de conteo. Elige y justifica una alternativa para continuar en `df_limpio` conserva los valores imputados sin redondearlos y usa el mismo tratamiento categórico del inciso anterior.

In [ ]:
# Desarrolla los incisos 1.1 a 1.4; puedes agregar celdas.

### Interpretación del ejercicio 1
Escribe aquí tus conclusiones y la justificación de la imputación elegida.

---
## Ejercicio 2: Generación y codificación de características (35 pts)

Trabaja con `df_limpio` obtenido en el ejercicio anterior, no uses la variable `passed` para construir las características.

**2.1 (10 pts).** Verifica que `absences` sea no negativa y genera una nueva variable: `absences_log = np.log1p(absences)`, conservando `absences`. Genera `age_group` mediante la columna `age` con intervalos **[0,17), [17,19), [19,21), [21,infinito)**, etiquetados `menor_17`, `17_18`, `19_20`, `21_mas`, en ese orden. Conserva `age` en el dataframe. Verifica manualmente cinco filas, explica qué efecto tiene el logaritmo sobre los valores altos en la variable `absences_log` y qué información se pierde al agrupar edades. No redondees una edad imputada antes de asignar su grupo.

**2.2 (5 pts).** Codifica `higher` e `internet` con `no=0`, `yes=1`, `sex` con `F=0`, `M=1` y elige una codificación categorica ordinal para `age_group` (lineal, cuadrática). Conserva `studytime` como código ordinal del 1 al 4. Explica por qué sus diferencias numéricas no garantizan distancias iguales en horas de estudio. ¿Qué codificación podrías recomendar para esta variable?.

**2.3 (10 pts).** Aplica one-hot encoding a `Mjob` y `reason`, sin eliminar categoría de referencia. Reporta categorías, columnas creadas y número de columnas antes y después. Verifica que, para cada fila, la suma de los indicadores de cada variable nominal sea 1. Explica por qué sería incorrecto imponer un orden alfabético como escala numérica de ocupaciones o motivos de elección.

**2.4 (10 pts).** Integra los predictores codificados en `X_codificado` y guarda las etiquetas originales de `passed` en una Serie `y`. Elimina las columnas categóricas reemplazadas, conserva el índice de las filas y confirma que todos los predictores sean numéricos y que `passed` no esté en `X_codificado`. Muestra frecuencias y proporciones de las dos clases.

In [ ]:
# Desarrolla los incisos 2.1 a 2.4.

### Interpretación del ejercicio 2
Explica tus decisiones de generación y codificación.

---
## Ejercicio 3: Escalamiento, integración y validación (35 pts)

**3.1 (10 pts).** Sobre copias independientes de `X_codificado`, compara `StandardScaler` y `MinMaxScaler` para **`age`, `absences`, `absences_log`**. No escales indicadores, `higher`, `internet`, `sex`, `age_group` ni `studytime`. Construye una tabla por método y columna con mínimo, máximo, media y desviación estándar poblacional (`ddof=0`). Explica las diferencias entre estandarizar y Min-Max.

**3.2 (10 pts).** Elige uno de los métodos anteriores o `RobustScaler` y justifícalo con las distribuciones, asimetría, curtósis y los valores extremos observados. Obtén `X_final` a partir de los datos elegidos anteriormente. Verifica la propiedad del método elegido: media 0 y desviación 1 para estandarización, mínimo 0 y máximo 1 para Min-Max, o mediana 0 para Robust.

**3.3 (10 pts).** Con comprobaciones explícitas, verifica: 
- 649 filas índices de `X_final` y `y` alineados.
- Nombres de columnas únicos, ausencia de nulos y valores infinitos.
- Variable objetivo fuera de `X_final`
- Variables binarias/indicadores todavía en {0,1}.
- Exporta `salidas/X_final.csv` y `salidas/y_final.csv`, sin índice, preservando el mismo orden de filas.

**3.4 (5 pts).** Redacta una conclusión de 5 a 8 líneas sobre los cambios realizados y sus limitaciones. Explica por qué, si luego se evaluara un modelo con entrenamiento/prueba, los parámetros de imputación y escalamiento y las categorías del codificador deben aprenderse solo con entrenamiento. 

**Nota:** En este examen se prepara el modelo de datos completo, sin entrenar ningún algoritmo predictivo.


In [ ]:
# Desarrolla los incisos 3.1 a 3.3 y exporta los resultados.

### Conclusión final (inciso 3.4)
Escribe aquí tu conclusión.